# 🚨 D-Day Deadline Sprint — Qwen2.5-Coder-14B (bf16) Full SFT

**목적**: 마감 10시간 전, OOF 앙상블을 생략하고 전체 데이터를 학습한 단일 모델로 빠르게 추론하여 최고 성능의 제출 파일 생성.

**필수 업로드 파일:**
- `my_code_0514from0508/` 폴더 전체 (방금 AI가 수정한 14B bf16 최신본 압축 파일)
- `data/train.csv`, `data/test.csv`, `data/somenna_submission.csv`

In [ ]:
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps unsloth_zoo
!pip install --upgrade "transformers==5.5.0" "trl==0.24.0" "datasets<4.4.0"
!pip install cut_cross_entropy hf_transfer msgspec tyro peft accelerate bitsandbytes xformers
!pip install flash-attn --no-build-isolation
!pip install pandas tqdm scikit-learn sentence-transformers

In [ ]:
import torch
gpu_name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU: {gpu_name} | VRAM: {vram:.1f} GB')

In [ ]:
import os, re
os.chdir('/content')
SRC_DIR = '/content/my_code_0514from0508'
train_py = f'{SRC_DIR}/train.py'

# Validation 무시하고 전체 데이터 학습하도록 플래그 강제 수정
with open(train_py, 'r', encoding='utf-8') as f:
    src = f.read()
src = re.sub(r'FINAL_TRAIN_ON_FULL_DATA\s*=\s*False', 'FINAL_TRAIN_ON_FULL_DATA = True', src)
with open(train_py, 'w', encoding='utf-8') as f:
    f.write(src)
print("✅ FINAL_TRAIN_ON_FULL_DATA flag patched to True!")

In [ ]:
import subprocess, sys
print("🚀 Starting Full Data SFT Training (Qwen2.5-Coder-14B bf16)... (Expected time: ~1 hour)")
result = subprocess.run([sys.executable, f'{SRC_DIR}/train.py'], cwd='/content')
print('Training Return code:', result.returncode)

In [ ]:
print("⚡ Starting Single Model Inference... (Expected time: ~1 hour)")
result = subprocess.run([sys.executable, f'{SRC_DIR}/inference.py'], cwd='/content')
print('Inference Return code:', result.returncode)

In [ ]:
from google.colab import files
import os
import pandas as pd

sub_path = '/content/submission.csv'
if os.path.exists(sub_path):
    print("🎉 submission.csv found!")
    df = pd.read_csv(sub_path)
    print(f"Total rows: {len(df)}")
    print("OP Distribution:", dict(df['op'].value_counts()))
    empty = (df['target_id'].astype(str).str.strip() == '').sum()
    print(f"Empty target_ids (Must be 0): {empty}")
    
    files.download(sub_path)
else:
    print("❌ submission.csv not found!")